# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/div828/Flyrank_starternotebook/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane (from Week 1): Refresh / Content Opportunity Scoring.**

My question -- "which pages should be reviewed first" -- is a **"which ones
first" question, so it's fundamentally a ranking/scoring task**, not a plain
yes/no classification task. The output I actually want is an ordered list:
the top N pages an editor should look at, given limited review time.

In practice I'll get there by training a **classifier** that predicts a
probability for a proxy label (see Section 2), then **sorting pages by that
probability** to build the queue. So under the hood this is
classification-to-produce-a-score, but the thing that matters for the decision
is the resulting rank, not the 0/1 label itself. This matches how the starter
pipeline in this repo is built (`02_baseline_score.py` -> `03_train_model.py` ->
ranked queue in `04_evaluate_and_export.py`).


In [1]:
# Framing only -- no computation needed for this section.
# (The model itself isn't trained until Week 5 / w05_model.ipynb.)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**This is a proxy, not a true observed future outcome -- and I want to be
upfront about that.**

The starter dataset gives me `trend_direction`, a bucket computed from
`trend_pct` over the trailing 90 days. The proxy I'll use is:

```
is_declining_proxy = (trend_direction == "down")
```

This is a *proxy* because it's a label about the **current** trailing window,
not something measured in a **future** window after a decision point. A
stronger, leakage-safer version (available in the warehouse release, not the
starter CSV) would be: features from a prior 90-day window -> decline measured
in the *next* 30 days. I don't have that forward-looking window in the starter
CSV, so for now I'm using the current-window proxy the same way the starter
pipeline does, and I'll flag it as a proxy everywhere I mention it.

**Leakage check:** `trend_direction` and `trend_pct` are literally what the
label is built from, so they can **never** be features -- only the label. I'll
exclude both from any feature set going forward.


In [2]:
# Framing only -- the actual target column is built and shown in Section 4.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@K (specifically Precision@50).**

Accuracy doesn't fit here -- an editor doesn't review "all pages classified
positive," they review a fixed-size list because their time is capped.
Precision@50 answers the actual question this decision needs: **of the top 50
pages the queue puts in front of an editor, how many are genuinely worth their
time?**

This connects directly to the Week 1 cost analysis: a low Precision@50 means
editors spend real hours on pages that weren't actually declining (wasted
review capacity -- the false-positive cost from Week 1). It doesn't fully
capture the false-negative cost (a real decline that never makes the top 50),
so later weeks I'll also want to look at recall/coverage -- but for a
capacity-limited weekly review queue, Precision@K is the metric that matches
how the output actually gets used.


In [3]:
# Framing only -- no computation needed for this section.
# Precision@50 will actually be computed once a model exists (Week 5+).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one pseudonymized content item (`content_id`), with its trailing
90-day content and search metrics. Below I load the real starter CSV, keep the
columns relevant to this lane, and sketch the proxy target column defined in
Section 2.


In [5]:
import pandas as pd

df = pd.read_csv(
    "https://raw.githubusercontent.com/div828/Flyrank_starternotebook/main/data/raw/content_refresh_anonymized.csv"
)

# proxy target, defined in Section 2 -- never used as a feature itself
df["is_declining_proxy"] = (df["trend_direction"] == "down")

lane_cols = [
    "content_id", "client_id",
    "impressions_90d", "sessions_90d", "clicks_90d",
    "content_age_days", "days_since_last_update",
    "word_count", "avg_position", "ctr",
    "engagement_rate", "scroll_rate",
    "trend_direction",
    "is_declining_proxy",
]

unit_of_analysis = df[lane_cols]

print("One row = one content item (content_id), trailing 90-day metrics")
print("Shape:", unit_of_analysis.shape)

unit_of_analysis.head(8)

One row = one content item (content_id), trailing 90-day metrics
Shape: (30000, 14)


,content_id,client_id,impressions_90d,sessions_90d,clicks_90d,content_age_days,days_since_last_update,word_count,avg_position,ctr,engagement_rate,scroll_rate,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,17,29,187,20,3221.0,10.6,0.76,5.88,4.55,down,True
1,content_a1fb4e703a9e,client_4e07408562,15320,9,7,445,25,2481.0,20.3,0.05,0.00,10.00,down,True
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,141,20,3515.0,36.5,0.09,0.00,28.57,down,True
3,content_331d6c4de07b,client_19581e27de,11751,78,58,463,22,NaN,6.2,0.49,1.28,3.45,stable,False
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,24,263,14,2803.0,44.0,0.13,0.00,24.29,down,True
5,content_d4084a4bc775,client_f369cb89fc,3970,5,1,147,20,3080.0,8.5,0.03,0.00,25.00,down,True
6,content_9a34b442b552,client_8722616204,20,1,0,90,20,3059.0,7.0,0.00,0.00,0.00,down,True
7,content_a63219c6e95a,client_19581e27de,1724,28,1,445,22,NaN,21.2,0.06,3.57,7.14,stable,False


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single threshold struggles here because several signals interact instead of
acting independently. A page can be stale (`days_since_last_update`) but still
have strong demand (`impressions_90d`) -- that combination matters more than
either signal alone. Similarly, a low `ctr` only means something once you
adjust for `avg_position`, and `engagement_rate` vs `scroll_rate` can disagree
with each other in ways a single if-statement can't weigh sensibly.

The starter pipeline in this repo already tested this on this exact proxy
label: the hand-written rule baseline gets Precision@50 ~ 0.24, while a random
forest gets ~ 0.74 (see `outputs/model_report.md`). That's a real result on
this data, but it's for this proxy label specifically -- **I'm treating it as a
hypothesis to re-test on my own splits and validation, not as proof that ML
will automatically win for whatever exact target I finalize.** A model may be
able to learn which *combinations* of staleness, demand, position, and
engagement matter together -- something a single fixed rule can't easily
express.


In [ ]:
# Framing only. Model training and the actual precision@50 comparison
# on my own validation split happen starting in w05_model.ipynb.
